In [1]:
!nvidia-smi

Wed Mar 11 16:00:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install ultralytics
!pip install kaggle
!pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.9 MB/s eta 0:00:00


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"sidrashaikh1407","key":"a0553f1a1f8108ad5d349a90e60e5ee6"}'}

In [6]:
import os

os.makedirs('/root/.kaggle', exist_ok=True)

!mv kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json

In [7]:
!kaggle datasets list

ref                                                      title                                                  size  lastUpdated                 downloadCount  voteCount  usabilityRating  
-------------------------------------------------------  -----------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
dmahajanbe23/bmw-global-automotive-sales                 BMW Global Automotive Sales                           55017  2026-02-22 18:18:38.170000           5882        117  1.0              
thuandao/superstore-sales-analytics                      SuperStore Sales Analytics                          2283450  2026-03-06 12:31:25.800000           1565         38  1.0              
syedaeman2212/airline-ticket-prices-dataset              Airline Ticket Prices Dataset                          4409  2026-03-05 16:30:06.230000            907         33  1.0              
grandmaster07/student-exam-performance-dataset-ana

In [8]:
!kaggle datasets download -d imsparsh/deepweeds

Dataset URL: https://www.kaggle.com/datasets/imsparsh/deepweeds
License(s): Attribution 4.0 International (CC BY 4.0)
 87% 408M/470M [00:00<00:00, 493MB/s]
100% 470M/470M [00:00<00:00, 499MB/s]


In [9]:
!unzip deepweeds.zip

Streaming output truncated to the last 5000 lines.
  inflating: images/20180109-072434-2.jpg  
  inflating: images/20180109-072438-1.jpg  
  inflating: images/20180109-072442-2.jpg  
  inflating: images/20180109-072448-1.jpg  
  inflating: images/20180109-072451-2.jpg  
  inflating: images/20180109-072501-2.jpg  
  inflating: images/20180109-072502-1.jpg  
  inflating: images/20180109-072510-2.jpg  
  inflating: images/20180109-072512-1.jpg  
  inflating: images/20180109-072518-2.jpg  
  inflating: images/20180109-072524-1.jpg  
  inflating: images/20180109-072527-2.jpg  
  inflating: images/20180109-072534-1.jpg  
  inflating: images/20180109-072536-2.jpg  
  inflating: images/20180109-072543-1.jpg  
  inflating: images/20180109-072554-2.jpg  
  inflating: images/20180109-072555-1.jpg  
  inflating: images/20180109-072603-2.jpg  
  inflating: images/20180109-072606-1.jpg  
  inflating: images/20180109-072615-2.jpg  
  inflating: images/20180109-072622-1.jpg  
  inflating: images/20180

In [10]:
!ls

deepweeds.zip  drive  images  labels  sample_data


In [11]:
import pandas as pd
import os

df = pd.read_csv("labels/labels.csv")

os.makedirs("yolo_labels", exist_ok=True)

for _, row in df.iterrows():

    filename = row["Filename"]
    class_id = row["Label"]

    txt = filename.replace(".jpg",".txt")

    with open(f"yolo_labels/{txt}", "w") as f:
        f.write(f"{class_id} 0.5 0.5 1 1")

In [12]:
import random
import shutil
from pathlib import Path
import os

images = list(Path("images").glob("*.jpg"))

random.shuffle(images)

split = int(0.8 * len(images))

train = images[:split]
val = images[split:]

os.makedirs("dataset/train/images", exist_ok=True)
os.makedirs("dataset/train/labels", exist_ok=True)

os.makedirs("dataset/val/images", exist_ok=True)
os.makedirs("dataset/val/labels", exist_ok=True)

for img in train:

    label = Path("yolo_labels")/(img.stem+".txt")

    shutil.copy(img,"dataset/train/images/")
    shutil.copy(label,"dataset/train/labels/")

for img in val:

    label = Path("yolo_labels")/(img.stem+".txt")

    shutil.copy(img,"dataset/val/images/")
    shutil.copy(label,"dataset/val/labels/")

In [13]:
data_yaml = """
train: /content/dataset/train/images
val: /content/dataset/val/images

nc: 8

names:
- chinee_apple
- lantana
- parkinsonia
- parthenium
- prickly_acacia
- rubber_vine
- siam_weed
- snake_weed
"""

with open("deepweeds.yaml","w") as f:
    f.write(data_yaml)

In [14]:
import torch

torch.backends.cudnn.benchmark = True

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [15]:
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [16]:
model = YOLO("yolov8n.pt")

In [17]:
model.train(
    data="deepweeds.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    amp=True,
    workers=4,
    project="/content/drive/MyDrive/yolo_training",
    name="yolov8n"
)

Streaming output truncated to the last 5000 lines.
train: /content/dataset/train/images/20180112-071336-2.jpg: ignoring corrupt image/label: Label class 8 exceeds dataset class count 8. Possible class labels are 0-7
train: /content/dataset/train/images/20180112-071354-2.jpg: ignoring corrupt image/label: Label class 8 exceeds dataset class count 8. Possible class labels are 0-7
train: /content/dataset/train/images/20180112-071355-2.jpg: ignoring corrupt image/label: Label class 8 exceeds dataset class count 8. Possible class labels are 0-7
train: /content/dataset/train/images/20180112-071404-2.jpg: ignoring corrupt image/label: Label class 8 exceeds dataset class count 8. Possible class labels are 0-7
train: /content/dataset/train/images/20180112-071421-2.jpg: ignoring corrupt image/label: Label class 8 exceeds dataset class count 8. Possible class labels are 0-7
train: /content/dataset/train/images/20180112-071430-2.jpg: ignoring corrupt image/label: Label class 8 exceeds dataset clas

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x78d8d1890560>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,

In [18]:
!pip install thop

In [19]:
from ultralytics import YOLO
import torch

model = YOLO("/content/drive/MyDrive/yolo_training/yolov8n/weights/best.pt")

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C2f(
        (cv1): Conv(
          (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(48, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_s

In [20]:
from thop import profile

dummy = torch.randn(1,3,640,640).to(device)

flops, params = profile(model.model, inputs=(dummy,), verbose=False)

params_M = params / 1e6
flops_G = flops / 1e9

print("Params (M):", params_M)
print("FLOPs (G):", flops_G)

Params (M): 3.012408
FLOPs (G): 4.1008384


In [21]:
import time

runs = 200

torch.cuda.synchronize()
start = time.time()

for _ in range(runs):
    model.model(dummy)

torch.cuda.synchronize()
end = time.time()

fps = runs / (end - start)

print("FPS:", fps)

FPS: 82.13452028760665


In [22]:
import subprocess
import numpy as np

def get_gpu_power():
    result = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=power.draw", "--format=csv,noheader,nounits"]
    )
    return float(result.decode().strip())

power_samples = []

for _ in range(50):
    power_samples.append(get_gpu_power())

avg_power = np.mean(power_samples)  # Watts

energy_per_frame = avg_power / fps

print("Energy (J/frame):", energy_per_frame)

Energy (J/frame): 0.3956923335790634


In [23]:
metrics = model.val()

map50 = metrics.box.map50
map5095 = metrics.box.map
recall = metrics.box.r

print("mAP@0.5:", map50)
print("mAP@0.5:0.95:", map5095)
print("Weed Recall:", recall)

Model summary (fused): 73 layers, 3,007,208 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1214.3±601.7 MB/s, size: 28.7 KB)
val: Scanning /content/dataset/val/labels.cache... 3502 images, 0 backgrounds, 1814 corrupt: 100% ━━━━━━━━━━━━ 3502/3502 1.6Git/s 0.0s
val: /content/dataset/val/images/20170128-102458-0.jpg: ignoring corrupt image/label: Label class 8 exceeds dataset class count 8. Possible class labels are 0-7
val: /content/dataset/val/images/20170128-102642-0.jpg: ignoring corrupt image/label: Label class 8 exceeds dataset class count 8. Possible class labels are 0-7
val: /content/dataset/val/images/20170128-102719-0.jpg: ignoring corrupt image/label: Label class 8 exceeds dataset class count 8. Possible class labels are 0-7
val: /content/dataset/val/images/20170128-102943-0.jpg: ignoring corrupt image/label: Label class 8 exceeds dataset class count 8. Possible class labels are 0-7
val: /content/dataset/val/images/20170128-103141-0.jpg: i

In [24]:
small_ap = metrics.box.maps[0]

print("Small Weed AP:", small_ap)

Small Weed AP: 0.9929044670490164


In [25]:
print("\nModel Evaluation\n")

print("Params (M):", params_M)
print("FLOPs (G):", flops_G)
print("Energy (J/frame):", energy_per_frame)
print("FPS:", fps)
print("mAP@0.5:", map50)
print("mAP@0.5:0.95:", map5095)
print("Weed Recall:", recall)
print("Small Weed AP:", small_ap)


Model Evaluation

Params (M): 3.012408
FLOPs (G): 4.1008384
Energy (J/frame): 0.3956923335790634
FPS: 82.13452028760665
mAP@0.5: 0.9924802213963906
mAP@0.5:0.95: 0.9924600557444453
Weed Recall: [    0.94872     0.96308     0.99145           1     0.96385     0.99482     0.98161     0.94499]
Small Weed AP: 0.9929044670490164


In [26]:
import pandas as pd

# Create dataframe
results = pd.DataFrame({
    "Model": ["YOLOv8n"],
    "Params (M)": [params_M],
    "FLOPs (G)": [flops_G],
    "Energy (J/frame)": [energy_per_frame],
    "FPS": [fps],
    "mAP@0.5": [map50],
    "mAP@0.5:0.95": [map5095],
    "Weed Recall": [recall],
    "Small Weed AP": [small_ap]
})

# Save CSV
results.to_csv("/content/drive/MyDrive/yolo_training/model_metrics.csv", index=False)

print("CSV saved successfully")
print(results)

CSV saved successfully
     Model  Params (M)  FLOPs (G)  Energy (J/frame)       FPS  mAP@0.5  \
0  YOLOv8n    3.012408   4.100838          0.395692  82.13452  0.99248   

   mAP@0.5:0.95                                        Weed Recall  \
0       0.99246  [0.9487179487179487, 0.9630773853708195, 0.991...   

   Small Weed AP  
0       0.992904  
